# SPHEREx change detection on Planet X v1.3

Run from a local repository clone. The EDA and measured results below load from the repository. The final cell can reproduce both experiments with an extracted dataset.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import pandas as pd
from IPython.display import Image, display
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "v13_experiment.py").is_file())
run = repo / "experiments/v13"
eda = run / "dataset_overlay/eda"
display(Image(filename=str(run / "eda.png")))
display(pd.read_csv(eda / "split_summary.csv").query("mission == 'spherex'"))

## Candidate classification

Compare validation F1 across image and tree weights. The frozen operating point controls which candidates enter review. Accuracy counts positive and background candidates together.

In [ ]:
results = json.loads((run / "ensemble.json").read_text())
display(pd.DataFrame(results["validation"]).T.rename_axis("CNN weight"))
display(pd.Series(results["test"],name="test"))
display(pd.read_csv(run/"ensemble_confusion.csv",index_col=0))

## Recovery on injections and original fields

Each synthetic object receives at most one recovery credit. The catalogue audit counts frames with a correctly classified alert within two pixels of a named real source.

In [ ]:
display(pd.read_csv(run/"ensemble_objects.csv"))
display(pd.read_csv(run/"ensemble_targets.csv"))
display(pd.read_csv(run/"ensemble_alerts.csv.gz").nlargest(10,"priority"))

## Repeat training and inference

Set the dataset directory and the switch below. Use a fresh cache path for a fresh extraction. The run uses the existing CNN weights and retrains the boosted trees.

In [ ]:
retrain = False
dataset = repo / "data/planetx-dataset-v1.3.0"
cache = dataset.parent / "v13-candidates.parquet"
if retrain:
    for script in ["v13_experiment.py", "v13_ensemble.py"]:
        subprocess.run([sys.executable, script, str(dataset), str(cache)], cwd=repo, check=True)